# Lab 04 — Build a Tiny Decoder-Only Transformer
**Goal:** Implement a minimal causal self-attention block and train it on a tiny corpus.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
T,C=8,32
x=torch.randn(2,T,C)
print("input shape:", x.shape)


## Predict before running
Before running, predict:
- What shape should Q, K, and V have?
- Why must future-token probabilities be zero?
- Why does every attention row sum to one?


In [ ]:
class TinyAttention(nn.Module):
    def __init__(self,C,heads=4):
        super().__init__()
        self.qkv=nn.Linear(C,3*C,bias=False)
        self.proj=nn.Linear(C,C,bias=False)
        self.h=heads
        self.register_buffer("mask",torch.tril(torch.ones(T,T)).bool())
    def forward(self,x):
        B,Tc,Cc=x.shape
        q,k,v=self.qkv(x).chunk(3,dim=-1)
        q=q.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        k=k.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        v=v.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        a=(q@k.transpose(-2,-1))/(Cc//self.h)**0.5
        a=a.masked_fill(~self.mask[:Tc,:Tc],float("-inf"))
        a=F.softmax(a,dim=-1)
        y=(a@v).transpose(1,2).contiguous().view(B,Tc,Cc)
        return self.proj(y)
m=TinyAttention(C)
y=m(x)
print("output:",y.shape)


## Explain the result
The causal mask preserves autoregressive training: position t can use positions ≤ t, never the future target. Softmax converts scores into a distribution, while the head dimension determines the matrix-multiplication cost.


In [ ]:
# Inspect one attention matrix row. Each row should sum to ~1 and must not put mass on future tokens.
with torch.no_grad():
    q,k,v=m.qkv(x).chunk(3,dim=-1)
    q=q.view(2,T,m.h,C//m.h).transpose(1,2)
    k=k.view(2,T,m.h,C//m.h).transpose(1,2)
    a=(q@k.transpose(-2,-1))/(C//m.h)**0.5
    a=a.masked_fill(~m.mask[:T,:T],float("-inf"))
    a=F.softmax(a,dim=-1)
print("row sums:", a[0,0].sum(-1))
print("future attention in last row:", a[0,0,-1].tolist())


## Critical-thinking answers
The important engineering skill is shape accounting. A Transformer is many matrix operations composed with residual paths; errors often appear first as dimension mismatches or incorrect masking. Learn to reason with (batch, sequence, hidden, heads, head_dim) before relying on framework abstractions.

### Sources
https://arxiv.org/abs/1706.03762
